# Token Classification (NER)

> Named Entity Recognition, POS tagging, and other token-level tasks

This notebook demonstrates how to fine-tune a HuggingFace transformer for token classification using blurr.

In [ ]:
#| default_exp text.token_classification

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from blurr.text.token_classification import TokenClassificationLearner

## Named Entity Recognition (CoNLL-2003)

### Load from HuggingFace Dataset

In [ ]:
learn = TokenClassificationLearner.from_hf(
    'conll2003',
    model_name='prajjwal1/bert-tiny',
    tokens_col='tokens',
    labels_col='ner_tags',
    max_samples=200,
    max_length=128,
    bs=16,
)

### Training

Entity-level metrics (precision, recall, F1) are automatically added via seqeval.

In [ ]:
learn.fit_one_cycle(1, 2e-5)

### Prediction

Predict named entities in raw text. Entities are automatically aggregated from subword tokens.

In [ ]:
results = learn.blurr_predict_tokens('Hugging Face is based in New York City.')
for entity in results[0]:
    print(f"  {entity['entity_group']}: '{entity['word']}' (score: {entity['score']:.4f})")

In [ ]:
# Different aggregation strategies:
# 'none'    — raw token-level predictions
# 'simple'  — group consecutive same-type entities (default)
# 'first'   — score = first subword token's score
# 'max'     — score = max of all subword tokens
# 'average' — score = mean of all subword tokens

results = learn.blurr_predict_tokens(
    'Apple Inc. was founded by Steve Jobs in Cupertino, California.',
    aggregation_strategy='simple',
)
for entity in results[0]:
    print(entity)

## From a DataFrame

For custom NER data, create a DataFrame with columns of token and label lists:

In [ ]:
import pandas as pd

data = pd.DataFrame({
    'tokens': [
        ['John', 'lives', 'in', 'New', 'York', '.'],
        ['Apple', 'is', 'a', 'company', '.'],
        ['Paris', 'is', 'the', 'capital', 'of', 'France', '.'],
        ['Mary', 'works', 'at', 'Google', '.'],
    ],
    'labels': [
        ['B-PER', 'O', 'O', 'B-LOC', 'I-LOC', 'O'],
        ['B-ORG', 'O', 'O', 'O', 'O'],
        ['B-LOC', 'O', 'O', 'O', 'O', 'B-LOC', 'O'],
        ['B-PER', 'O', 'O', 'B-ORG', 'O'],
    ],
})

learn = TokenClassificationLearner.from_df(
    data,
    'prajjwal1/bert-tiny',
    tokens_col='tokens',
    labels_col='labels',
    bs=2,
    max_length=64,
)

## Key Features

- **Subword alignment**: Labels are automatically aligned with subword tokens
- **Entity-level metrics**: Precision, recall, F1 via seqeval (not just token accuracy)
- **Aggregation strategies**: Multiple strategies for combining subword predictions
- **Any HF model**: Works with BERT, RoBERTa, DeBERTa, XLNet, etc.